# Running on Slurm

You already saw the lecture; keep these essentials handy:
- Slurm controllers (`slurmctld`, `slurmd`, `slurmdbd`) coordinate resource requests, while `sbatch` sends non-interactive jobs and `srun` launches work inside an allocation.
- Every `#SBATCH` directive is a contract: request only what you need (`--time`, `--cpus-per-task`, `--mem`), keep the queue fair, and prefer job arrays for embarrassingly parallel sweeps.
- `--export=ALL` propagates your login environment to the job so that module/conda activation and custom variables remain consistent across the cluster nodes.


## Minimal parameterized run (`scripts/run_local.py`)
- Reads `config/params.yaml` to capture data size, signal parameters, and noise level; extra configs generated by sweeps drop in seamlessly.
- Logs the current git commit via `scripts/capture_git.py`, guaranteeing that every output folder ties back to an exact revision.
- Generates synthetic `y = m x + b + noise` data, fits a straight line with NumPy, and saves both `data.csv` and `results.json`.
- Writes artifacts to `outputs/{git}-{timestamp}` with a fresh log file and a copied config so runs remain immutable.


In [ ]:
from pathlib import Path
from IPython.display import Code, display

display(Code(data=Path('scripts/run_local.py').read_text(), language='python'))

### Default parameter file (`config/params.yaml`)
Override values by handing `--config` a generated YAML from the sweep or your own custom file.

In [ ]:
from pathlib import Path
from IPython.display import Code, display

display(Code(data=Path('config/params.yaml').read_text(), language='yaml'))

## Parameter sweep launcher (`scripts/run_param_sweep.py`)
- Consumes `config/sweep.yaml`, computes the cartesian product of the provided parameter lists, and merges each combo with the base config.
- Writes per-run YAMLs to `config/generated_params/` plus a newline-delimited `config/param_grid.txt` that Slurm arrays can replay.
- `--dry-run` prints the exact `python scripts/run_local.py --config ...` commands without touching the filesystem, letting you sanity-check before heavy submissions.


In [ ]:
from pathlib import Path
from IPython.display import Code, display

display(Code(data=Path('scripts/run_param_sweep.py').read_text(), language='python'))

### Sweep description (`config/sweep.yaml`)
This example creates 3 (slope) × 2 (intercept) = 6 jobs, which lines up with the array definition used below.

In [ ]:
from pathlib import Path
from IPython.display import Code, display

display(Code(data=Path('config/sweep.yaml').read_text(), language='yaml'))

## Slurm job templates
### Single run (`scripts/slurm_job.sbatch`)
- Requests modest resources, sets per-job stdout/stderr files through `#SBATCH --output/--error`, **and** mirrors logs locally via `exec > >(tee ...)`.
- Creates `logs/` safely before writing, exports the login environment (`--export=ALL`), and activates your preferred module/conda stack before calling `python scripts/run_local.py --config ...`.


In [ ]:
from pathlib import Path
from IPython.display import Code, display

display(Code(data=Path('scripts/slurm_job.sbatch').read_text(), language='bash'))

### Array run (`scripts/slurm_array_job.sbatch`)
- Uses `#SBATCH --array=0-5` to fan out across the 6 commands stored in `config/param_grid.txt`.
- Reads the command corresponding to `$SLURM_ARRAY_TASK_ID`, echoes it for traceability, and then executes it after the same safe logging setup.
- Because the grid file is written by `run_param_sweep.py`, you can regenerate it anytime (or keep multiple versions by pointing `GRID_FILE` at a different path).


In [ ]:
from pathlib import Path
from IPython.display import Code, display

display(Code(data=Path('scripts/slurm_array_job.sbatch').read_text(), language='bash'))